In [1]:

from sqlalchemy import create_engine, text
import ssl

## CONEXION BBDD MYSQL ##
DB_USER = "nuclio"
DB_PASS = "nuclioTFM6"
DB_HOST = "nuclio.mysql.database.azure.com"
DB_NAME = "olist"

# Crear engine apuntando a la base 'olist'
engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:3306/{DB_NAME}?charset=utf8mb4",
    pool_pre_ping=True,
    connect_args={"ssl": {"cert_reqs": ssl.CERT_NONE, "check_hostname": False}} 
)

# tablas 'olist'
with engine.connect() as conn:
    tables = conn.execute(text("SHOW TABLES")).fetchall()
    tables = [row[0] for row in tables]   # convertir a lista simple de strings
    
    print("Tablas en la base 'olist':")
    for t in tables:
        print("-", t)

Tablas en la base 'olist':
- dash_olist_categorias_resumen
- dash_olist_demorados
- dash_olist_sellers
- dash_olist_states
- dash_olist_ventas_meses
- dash_sentiment_analysis
- distribucion_pedidos
- olist_customers_dataset
- olist_geolocation_dataset
- olist_order_items_dataset
- olist_order_payments_dataset
- olist_order_reviews_dataset
- olist_orders_dataset
- olist_products_dataset
- olist_sellers_dataset
- pedidos_por_tiempo
- product_category_name_translation


In [2]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

import numpy, pandas, torch
print(numpy.__version__, pandas.__version__, torch.__version__)


Looking in indexes: https://download.pytorch.org/whl/cpu
Note: you may need to restart the kernel to use updated packages.
1.26.4 2.3.3 2.9.0+cpu


In [ ]:
import pandas as pd

# CARGA DEL DATASET ORIGINAL
file_path = r"C:\Users\HY269\Documents\GitHub\Nuclio_TFM6\sentiment_analysis_results.csv"

df = pd.read_csv(file_path, encoding="utf-8")

print("Dataset cargado correctamente.")
print(f"Total de registros: {len(df):,}")
print(f"Columnas: {list(df.columns)}")


Dataset cargado correctamente.
Total de registros: 82
Columnas: ['id', 'review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp', 'sentiment_label', 'sentiment_score', 'sentiment_category', 'review_year']


In [ ]:
import pandas as pd
from IPython.display import display, Markdown

# Validar columnas necesarias
if "sentiment_category" not in df.columns:
    raise KeyError("Falta la columna 'sentiment_category' en el DataFrame.")
if "review_creation_date" not in df.columns:
    raise KeyError("Falta la columna 'review_creation_date' en el DataFrame.")

# Convertir la fecha y extraer el año
df["review_creation_date"] = pd.to_datetime(df["review_creation_date"], errors="coerce")
df["order_year"] = df["review_creation_date"].dt.year

# Agrupar por año y categoría
sentiment_summary = (
    df.groupby(["order_year", "sentiment_category"], dropna=False)
      .size()
      .reset_index(name="count")
)

# Mostrar resultados
display(Markdown("### Distribución de reseñas por año y categoría de sentimiento"))
display(sentiment_summary.head(10))

print("\n DataFrame preparado para Looker Studio (dimensiones: order_year, sentiment_category).")


### Distribución de reseñas por año y categoría de sentimiento

,order_year,sentiment_category,count
0,2016,Negativo,1
1,2016,Neutral,1
2,2016,Positivo,1
3,2017,Negativo,10
4,2017,Neutral,4
5,2017,Positivo,17
6,2018,Negativo,16
7,2018,Neutral,7
8,2018,Positivo,25



 DataFrame preparado para Looker Studio (dimensiones: order_year, sentiment_category).


In [ ]:
from sqlalchemy import text
from sqlalchemy.types import Integer, String

# --- Guardar DataFrame en la base de datos ---
table_name = "dash_sentiment_analysis"

# --- Eliminar tabla si existe (para evitar conflictos por reflexión) ---
with engine.begin() as conn:
    conn.execute(text(f"DROP TABLE IF EXISTS `{table_name}`;"))

# --- Definir tipos de columnas ---
dtype_map = {
    "order_year": Integer(),
    "sentiment_category": String(50),
    "count": Integer(),
}

# --- Crear y cargar tabla ---
sentiment_summary.to_sql(
    name=table_name,
    con=engine,
    if_exists="fail",    
    index=False,
    dtype=dtype_map,
    method="multi"
)

print(f"Tabla '{table_name}' creada y cargada correctamente en la base de datos.")

# --- Validación opcional: mostrar número de registros cargados ---
with engine.connect() as conn:
    result = conn.execute(text(f"SELECT COUNT(*) FROM `{table_name}`;"))
    total_rows = result.scalar()
    print(f"Total de registros insertados: {total_rows:,}")


Tabla 'dash_sentiment_analysis' creada y cargada correctamente en la base de datos.
Total de registros insertados: 9
